# Sports analysis — starter notebook

This notebook talks to your local SQLite databases through **DuckDB**, attached **read-only**. The `.db` files stay the source of truth — nothing here can mutate them.

Query an attached DB by its alias: `nba.player_game`, `nfl.games`, `pga.tournaments`, `betting.futures_odds`.

In [ ]:
import sportsdb
import pandas as pd

con = sportsdb.connect()   # attaches the core sports DBs, read-only
sportsdb.databases()

## What tables exist

One row per table across every attached database.

In [ ]:
sportsdb.tables()

## Validate before you trust

Per `data_explorer/CLAUDE.md`: always sanity-check a query against a known fact before believing a derived number. Here the top single-game scoring list should be led by Kobe's 81 (2006) and the other 70+ point games.

In [ ]:
sportsdb.q("""
    SELECT p.player_name, g.pts, g.game_date
    FROM nba.player_game g
    JOIN nba.players p USING (player_id)
    ORDER BY g.pts DESC
    LIMIT 5
""")

## Cross-database queries

DuckDB's superpower here: one SQL statement can span every attached DB. No exporting, no copying.

In [ ]:
sportsdb.q("""
    SELECT 'nba' AS sport, COUNT(*) AS rows FROM nba.games
    UNION ALL SELECT 'nfl', COUNT(*) FROM nfl.games
    UNION ALL SELECT 'pga (tournaments)', COUNT(*) FROM pga.tournaments
    ORDER BY rows DESC
""")

## Quick plot

NBA scoring has trended up with pace — let's see it.

In [ ]:
import matplotlib.pyplot as plt

scoring = sportsdb.q("""
    SELECT season, AVG(home_pts + away_pts) AS avg_total_pts
    FROM nba.games
    GROUP BY season
    ORDER BY season
""")
ax = scoring.plot(x='season', y='avg_total_pts', legend=False, figsize=(10, 4))
ax.set_title('NBA average total points per game, by season')
ax.set_ylabel('avg total points')
plt.tight_layout()

## Export results to Parquet

Cache a result set for fast reuse or sharing. DuckDB writes Parquet natively — no pandas round-trip needed.

In [ ]:
from pathlib import Path

Path('outputs').mkdir(exist_ok=True)
con.execute("""
    COPY (
        SELECT season, AVG(home_pts + away_pts) AS avg_total_pts
        FROM nba.games GROUP BY season ORDER BY season
    ) TO 'outputs/nba_scoring_by_season.parquet' (FORMAT parquet)
""")
print('wrote outputs/nba_scoring_by_season.parquet')

## Add another database

Edit the `MANIFEST` dict in `sportsdb.py` (one line per DB), then `sportsdb.connect(refresh=True)`.

Want the reactive 2026 style? Open the marimo twin of this notebook:
`uv run marimo edit marimo_sample.py`